In [1]:
import pandas as pd
import numpy as np

# reproducibility
np.random.seed(42)

# -----------------------------
# DATE RANGE
# -----------------------------

months = pd.date_range(
    start='2023-01-01',
    end='2024-12-01',
    freq='MS'
)

plants = ['Plant A', 'Plant B', 'Plant C']

# -----------------------------
# PLANT METADATA
# -----------------------------

plants_df = pd.DataFrame({
    'Plant': plants,
    'Location': ['Helsinki', 'Tampere', 'Turku'],
    'Plant_Type': ['Old', 'Modern', 'Transitioning']
})

# -----------------------------
# PRODUCTION DATA
# -----------------------------

production_data = []

for plant in plants:
    
    base_production = {
        'Plant A': 8500,
        'Plant B': 12000,
        'Plant C': 10000
    }[plant]
    
    for month in months:
        
        # seasonality
        seasonal_factor = 1 + 0.20 * np.sin(
            month.month / 12 * 2 * np.pi
        )
        
        # random operational variation
        noise = np.random.normal(0, 500)
        
        production = (
            base_production *
            seasonal_factor +
            noise
        )
        
        # maintenance downtime
        if np.random.rand() < 0.05:
            production *= 0.7
        
        production = max(production, 5000)
        
        production_data.append([
            month,
            plant,
            round(production, 0)
        ])

production_df = pd.DataFrame(
    production_data,
    columns=[
        'Month',
        'Plant',
        'Concrete_Production_m3'
    ]
)

# -----------------------------
# CEMENT USAGE
# -----------------------------

cement_data = []

for index, row in production_df.iterrows():
    
    production = row['Concrete_Production_m3']
    plant = row['Plant']
    month = row['Month']
    
    # kg cement per cubic meter
    if plant == 'Plant A':
        cement_per_m3 = np.random.uniform(320, 360)
        
    elif plant == 'Plant B':
        cement_per_m3 = np.random.uniform(280, 320)
        
    else:
        cement_per_m3 = np.random.uniform(300, 340)
    
    # gradual sustainability improvement
    months_passed = (
        (month.year - 2023) * 12 +
        month.month
    )
    
    cement_per_m3 -= months_passed * 0.5
    
    cement_used_tons = (
        production *
        cement_per_m3
    ) / 1000
    
    cement_data.append([
        month,
        plant,
        round(cement_per_m3, 1),
        round(cement_used_tons, 1)
    ])

cement_df = pd.DataFrame(
    cement_data,
    columns=[
        'Month',
        'Plant',
        'Cement_kg_per_m3',
        'Cement_Used_tons'
    ]
)

# -----------------------------
# ENERGY DATA
# -----------------------------

energy_data = []

for index, row in production_df.iterrows():
    
    production = row['Concrete_Production_m3']
    plant = row['Plant']
    month = row['Month']
    
    # electricity usage
    electricity_kwh = (
        production *
        np.random.uniform(3.5, 5.5)
    )
    
    # diesel usage
    if plant == 'Plant A':
        diesel_liters = (
            production *
            np.random.uniform(1.5, 2.2)
        )
        
    elif plant == 'Plant B':
        diesel_liters = (
            production *
            np.random.uniform(0.8, 1.4)
        )
        
    else:
        diesel_liters = (
            production *
            np.random.uniform(1.2, 1.8)
        )
    
    # renewable electricity %
    renewable_pct = np.random.uniform(0.15, 0.55)
    
    renewable_kwh = (
        electricity_kwh *
        renewable_pct
    )
    
    energy_data.append([
        month,
        plant,
        round(electricity_kwh, 0),
        round(renewable_kwh, 0),
        round(diesel_liters, 0)
    ])

energy_df = pd.DataFrame(
    energy_data,
    columns=[
        'Month',
        'Plant',
        'Electricity_kWh',
        'Renewable_Electricity_kWh',
        'Diesel_Liters'
    ]
)

# -----------------------------
# EMISSIONS CALCULATIONS
# -----------------------------

emissions_data = []

for i in range(len(production_df)):
    
    cement_emissions = (
        cement_df.loc[i, 'Cement_Used_tons']
        * 0.90
    )
    
    diesel_emissions = (
        energy_df.loc[i, 'Diesel_Liters']
        * 0.00268
    )
    
    grid_electricity = (
        energy_df.loc[i, 'Electricity_kWh']
        -
        energy_df.loc[i, 'Renewable_Electricity_kWh']
    )
    
    electricity_emissions = (
        grid_electricity
        * 0.00025
    )
    
    total_emissions = (
        cement_emissions +
        diesel_emissions +
        electricity_emissions
    )
    
    production = production_df.loc[
        i,
        'Concrete_Production_m3'
    ]
    
    emissions_intensity = (
        total_emissions /
        production
    )
    
    emissions_data.append([
        production_df.loc[i, 'Month'],
        production_df.loc[i, 'Plant'],
        round(cement_emissions, 2),
        round(diesel_emissions, 2),
        round(electricity_emissions, 2),
        round(total_emissions, 2),
        round(emissions_intensity, 4)
    ])

emissions_df = pd.DataFrame(
    emissions_data,
    columns=[
        'Month',
        'Plant',
        'Cement_CO2_tons',
        'Diesel_CO2_tons',
        'Electricity_CO2_tons',
        'Total_CO2_tons',
        'CO2_per_m3'
    ]
)

# -----------------------------
# EXPORT CSV FILES
# -----------------------------

plants_df.to_csv(
    'plants.csv',
    index=False
)

production_df.to_csv(
    'production.csv',
    index=False
)

cement_df.to_csv(
    'cement_usage.csv',
    index=False
)

energy_df.to_csv(
    'energy.csv',
    index=False
)

emissions_df.to_csv(
    'emissions.csv',
    index=False
)

print("Dataset successfully generated.")

Dataset successfully generated.
